In [0]:
import os, sys
sys.path.append(os.path.dirname(os.getcwd()))
from pyspark.sql import functions as F
from src.transform import clean_web_events, get_rejected_events

bronze = spark.table("workspace.bronze.web_events")
get_rejected_events(bronze).write.mode("overwrite").saveAsTable("workspace.silver.web_events_rejected")
clean_web_events(bronze).write.mode("overwrite").saveAsTable("workspace.silver.web_events_clean")

orders = spark.table("workspace.bronze.orders")
(orders
    .filter(F.col("o_orderkey").isNotNull() & F.col("o_custkey").isNotNull())
    .dropDuplicates(["o_orderkey"])
    .select("o_orderkey", "o_custkey", "o_orderstatus", "o_totalprice", "o_orderdate")
    .write.mode("overwrite").saveAsTable("workspace.silver.orders_clean"))

spark.sql("""
CREATE OR REPLACE TABLE workspace.silver.customer_events AS
SELECT e.event_id, e.custkey, e.event_type, e.amount, e.event_time,
       o.order_count, o.total_order_value
FROM workspace.silver.web_events_clean e
LEFT JOIN (
  SELECT o_custkey, COUNT(*) AS order_count, SUM(o_totalprice) AS total_order_value
  FROM workspace.silver.orders_clean
  GROUP BY o_custkey
) o ON e.custkey = o.o_custkey
""")